# Neural Architecture Search with Reinforcement Learning — Implementation / 구현

**Paper**: Zoph, B. & Le, Q. V. (2017). Neural Architecture Search with Reinforcement Learning. ICLR 2017. arXiv:1611.01578

This notebook implements a *miniaturized* version of NAS for pedagogy. We do **not** train real CIFAR-10/PTB child networks (that would cost ~22,400 GPU-days). Instead we:

1. Build a controller RNN that auto-regressively samples a tiny architecture description.
2. Score sampled architectures against a *toy synthetic reward* (we know the optimal architecture, so we can verify the controller is learning).
3. Run the REINFORCE update rule with an EMA baseline.
4. Compare RL-driven search against random search — exactly the control experiment of the paper (§4 control experiment 2).

이 노트북은 NAS의 *축소판*을 PyTorch로 구현합니다. 실제 CIFAR-10/PTB child network 학습은 ~22,400 GPU-day가 필요하므로, 다음 핵심 메커니즘만 재현합니다:

1. controller RNN이 architecture description token을 auto-regressive로 샘플링.
2. *합성 보상*으로 architecture를 평가 (최적 architecture가 알려져 있어 controller 학습 여부 검증 가능).
3. EMA baseline을 적용한 REINFORCE 업데이트.
4. RL 기반 탐색 vs random search 비교 — 논문 §4 control experiment 2와 동일한 비교.

In [ ]:
import math
import random
from collections import deque
from typing import List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
np.random.seed(0)
random.seed(0)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## Part 1: Search space and synthetic reward / 탐색 공간과 합성 보상

We define a 4-layer architecture where each layer is described by 2 tokens:
- Token type A: filter size choice ∈ {1, 3, 5, 7}
- Token type B: number of filters choice ∈ {16, 32, 64, 128}

Total search space: $(4 \times 4)^4 = 65{,}536$ architectures.

Synthetic reward: we *pretend* that the best architecture has filter sizes [3, 5, 5, 3] and filter counts [32, 64, 64, 32]. Reward is a smooth function decreasing with token-wise distance from this ideal — analogous to validation accuracy, but cheap to evaluate.

총 탐색 공간은 $(4 \times 4)^4 = 65{,}536$개. 합성 보상은 "이상적 architecture"와의 토큰별 거리에 기반한 매끄러운 함수로 정의 — 실제 validation accuracy의 대용물 역할.

In [ ]:
FILTER_SIZES = [1, 3, 5, 7]
FILTER_COUNTS = [16, 32, 64, 128]
NUM_LAYERS = 4
TOKENS_PER_LAYER = 2  # (size, count)
VOCAB_SIZE = len(FILTER_SIZES)  # 4 — both token types use same cardinality here
T = NUM_LAYERS * TOKENS_PER_LAYER  # total tokens = 8

IDEAL_SIZES_IDX = [1, 2, 2, 1]   # filter_sizes[1]=3, [2]=5
IDEAL_COUNTS_IDX = [1, 2, 2, 1]  # filter_counts[1]=32, [2]=64


def synthetic_reward(arch_tokens: List[int]) -> float:
    """Compute a smooth reward in [0, 1] for an architecture token sequence.

    The architecture is encoded as 8 token indices alternating size / count for
    each of 4 layers. Reward is 1 minus the average token-wise normalized
    distance from the ideal architecture, then squared to amplify the gap
    between near-optimal and optimal — analogous to the paper's reward shaping.

    Args:
        arch_tokens: List of 8 integer token indices in [0, 3].

    Returns:
        A float reward in [0, 1] where 1 means the architecture matches the ideal.
    """
    assert len(arch_tokens) == T
    ideal = []
    for layer in range(NUM_LAYERS):
        ideal.append(IDEAL_SIZES_IDX[layer])
        ideal.append(IDEAL_COUNTS_IDX[layer])
    distance = sum(abs(a - i) for a, i in zip(arch_tokens, ideal)) / (T * (VOCAB_SIZE - 1))
    return (1.0 - distance) ** 2


def decode_arch(arch_tokens: List[int]) -> List[Tuple[int, int]]:
    """Decode token indices into human-readable (filter_size, filter_count) pairs."""
    layers = []
    for layer in range(NUM_LAYERS):
        size = FILTER_SIZES[arch_tokens[2 * layer]]
        count = FILTER_COUNTS[arch_tokens[2 * layer + 1]]
        layers.append((size, count))
    return layers


# Sanity check: ideal architecture should give reward 1.0
ideal_tokens = [t for layer in range(NUM_LAYERS) for t in (IDEAL_SIZES_IDX[layer], IDEAL_COUNTS_IDX[layer])]
print(f"Ideal arch tokens: {ideal_tokens}")
print(f"Decoded: {decode_arch(ideal_tokens)}")
print(f"Reward: {synthetic_reward(ideal_tokens):.4f}")
print(f"Random arch reward: {synthetic_reward([0, 0, 0, 0, 0, 0, 0, 0]):.4f}")

## Part 2: Controller RNN / 컨트롤러 RNN

The controller is a small LSTM that auto-regressively predicts $T=8$ tokens. At each timestep it:
1. Consumes an embedding of the previous token (or a learned `<START>` for $t=0$).
2. Runs one LSTM step.
3. Outputs a softmax over the vocabulary, samples the next token.
4. Returns the log-probability of that token (needed for REINFORCE).

이 부분은 논문 Figure 2와 정확히 동일한 구조: 토큰 → embedding → LSTM 1-step → softmax → sample → 다음 토큰의 입력으로 feed back. 작동 원리만 보여주는 미니 버전입니다.

In [ ]:
class Controller(nn.Module):
    """Auto-regressive LSTM controller that samples architecture token sequences.

    This mirrors the paper's controller (§3.1, Figure 2) but is intentionally
    tiny: a single LSTM layer with a small hidden size, predicting T tokens of
    a single shared vocabulary. The real paper uses a 2-layer LSTM with 35
    hidden units and per-position softmax heads.

    Attributes:
        vocab_size: Number of token choices at every position.
        hidden_size: LSTM hidden state dimension.
        embedding: Shared input embedding for sampled tokens.
        lstm_cell: Single LSTM cell shared across timesteps.
        output_head: Linear projection to vocabulary logits.
        start_token: Learned embedding fed as input at t=0.
    """

    def __init__(self, vocab_size: int = VOCAB_SIZE, hidden_size: int = 32) -> None:
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.lstm_cell = nn.LSTMCell(hidden_size, hidden_size)
        self.output_head = nn.Linear(hidden_size, vocab_size)
        self.start_token = nn.Parameter(torch.zeros(1, hidden_size))

    def sample(self, num_tokens: int = T) -> Tuple[List[int], torch.Tensor]:
        """Sample a token sequence and return per-token log-probabilities.

        Args:
            num_tokens: Number of tokens to sample (architecture length).

        Returns:
            tokens: List of sampled integer token indices.
            log_probs: Tensor of shape [num_tokens] with log P(a_t | a_<t).
        """
        h = torch.zeros(1, self.hidden_size)
        c = torch.zeros(1, self.hidden_size)
        x = self.start_token
        tokens: List[int] = []
        log_probs = []
        for _ in range(num_tokens):
            h, c = self.lstm_cell(x, (h, c))
            logits = self.output_head(h)
            dist = torch.distributions.Categorical(logits=logits)
            action = dist.sample()
            log_probs.append(dist.log_prob(action))
            tokens.append(int(action.item()))
            x = self.embedding(action)
        return tokens, torch.cat(log_probs)


controller_demo = Controller()
demo_tokens, demo_logp = controller_demo.sample()
print(f"Sampled tokens: {demo_tokens}")
print(f"Decoded layers: {decode_arch(demo_tokens)}")
print(f"Sum log-prob: {demo_logp.sum().item():.4f}")
print(f"Reward of sampled arch: {synthetic_reward(demo_tokens):.4f}")

## Part 3: REINFORCE training loop / REINFORCE 학습 루프

Implements the paper's equation (3) literally:

$$\hat g = \frac{1}{m}\sum_{k=1}^{m}\sum_{t=1}^{T}\nabla_{\theta_c}\log P(a_t^{(k)} \mid a_{<t}^{(k)};\theta_c)\,(R_k - b)$$

with $m$ architectures per gradient step and $b$ an exponential moving average of past rewards.

We track the rolling-best architecture each iteration so we can compare RL with random search later. 논문의 식 (3)을 그대로 구현 — $m$개 architecture sample, EMA baseline, 매 step의 best architecture 추적.

In [ ]:
def train_controller(
    num_iterations: int = 400,
    samples_per_step: int = 4,
    lr: float = 5e-3,
    baseline_decay: float = 0.95,
    seed: int = 0,
) -> Tuple[List[float], List[float], List[List[int]]]:
    """Run REINFORCE training of the controller against the synthetic reward.

    Args:
        num_iterations: Number of gradient updates.
        samples_per_step: Number of architectures sampled per update (m).
        lr: Learning rate for ADAM.
        baseline_decay: EMA decay for the reward baseline (closer to 1 = slower).
        seed: RNG seed for reproducibility.

    Returns:
        mean_rewards: Mean reward at each iteration.
        best_rewards: Running best reward seen so far at each iteration.
        best_arch_per_step: Best architecture token sequence at each step.
    """
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    controller = Controller()
    optimizer = torch.optim.Adam(controller.parameters(), lr=lr)
    baseline = 0.0
    mean_rewards = []
    best_rewards = []
    best_arch_per_step = []
    running_best = -1.0
    running_best_arch = None
    for it in range(num_iterations):
        log_probs_per_arch = []
        rewards = []
        archs = []
        for _ in range(samples_per_step):
            tokens, log_probs = controller.sample()
            r = synthetic_reward(tokens)
            archs.append(tokens)
            rewards.append(r)
            log_probs_per_arch.append(log_probs.sum())
        rewards_tensor = torch.tensor(rewards, dtype=torch.float32)
        # Update EMA baseline AFTER reading current rewards to keep it slightly stale (matches paper).
        advantages = rewards_tensor - baseline
        baseline = baseline_decay * baseline + (1.0 - baseline_decay) * float(rewards_tensor.mean().item())
        log_prob_tensor = torch.stack(log_probs_per_arch)
        loss = -(log_prob_tensor * advantages).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        # Track running best.
        for tok, r in zip(archs, rewards):
            if r > running_best:
                running_best = r
                running_best_arch = tok
        mean_rewards.append(float(rewards_tensor.mean().item()))
        best_rewards.append(running_best)
        best_arch_per_step.append(running_best_arch)
    return mean_rewards, best_rewards, best_arch_per_step


rl_mean, rl_best, rl_archs = train_controller(num_iterations=400, samples_per_step=4)
print(f"Final mean reward (RL): {rl_mean[-1]:.4f}")
print(f"Best reward seen (RL): {rl_best[-1]:.4f}")
print(f"Best arch found: {decode_arch(rl_archs[-1])}")
print(f"Ideal arch: {decode_arch(ideal_tokens)}")

## Part 4: Random search baseline / 랜덤 서치 베이스라인

This reproduces the paper's Figure 6 control experiment in miniature: replace policy gradient with uniform random architecture sampling and compare under the same compute budget (same number of evaluated architectures).

논문 Figure 6의 control experiment를 축소판으로 재현 — 동일한 evaluation 횟수 아래에서 random sampling과 RL의 학습 곡선을 비교.

In [ ]:
def random_search(num_iterations: int, samples_per_step: int, seed: int = 1) -> Tuple[List[float], List[float]]:
    """Sample architectures uniformly at random and track mean and running-best reward.

    Matches the compute budget (num_iterations * samples_per_step) used by
    train_controller, enabling a like-for-like comparison.
    """
    rng = random.Random(seed)
    mean_rewards = []
    best_rewards = []
    running_best = -1.0
    for _ in range(num_iterations):
        rewards = []
        for _ in range(samples_per_step):
            tokens = [rng.randrange(VOCAB_SIZE) for _ in range(T)]
            r = synthetic_reward(tokens)
            rewards.append(r)
            if r > running_best:
                running_best = r
        mean_rewards.append(float(np.mean(rewards)))
        best_rewards.append(running_best)
    return mean_rewards, best_rewards


rs_mean, rs_best = random_search(num_iterations=400, samples_per_step=4)
print(f"Final mean reward (random): {rs_mean[-1]:.4f}")
print(f"Best reward seen (random): {rs_best[-1]:.4f}")

## Part 5: Visualize RL vs random search / RL과 랜덤 서치 비교 시각화

Two diagnostic plots:
- **Mean reward per step:** the average quality of architectures sampled at each iteration. Random search is flat (uniform distribution); RL should rise.
- **Running-best reward:** monotone curve showing the best architecture seen so far.

이 그래프가 논문 Figure 6과 정확히 같은 메시지를 전달합니다 — RL은 평균 품질이 시간에 따라 상승하지만, random search는 평탄.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
iters = np.arange(len(rl_mean))
axes[0].plot(iters, rl_mean, label='RL (REINFORCE)', color='C0')
axes[0].plot(iters, rs_mean, label='Random search', color='C3', alpha=0.7)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Mean reward over m=4 sampled archs')
axes[0].set_title('Mean reward per step / 단계별 평균 보상')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(iters, rl_best, label='RL (REINFORCE)', color='C0')
axes[1].plot(iters, rs_best, label='Random search', color='C3', alpha=0.7)
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Running best reward')
axes[1].set_title('Best architecture so far / 누적 최고 보상')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nRL final running best:    {rl_best[-1]:.4f}")
print(f"Random final running best: {rs_best[-1]:.4f}")
print(f"RL final mean reward:    {rl_mean[-1]:.4f}")
print(f"Random final mean reward: {rs_mean[-1]:.4f}")

## Part 6: Inspect what the controller learned / 컨트롤러가 무엇을 배웠는지

Sample 1,000 architectures from the trained controller and check the marginal token distributions per position. If REINFORCE is working, the distribution at each position should peak at the ideal token index.

학습된 controller에서 1,000개 architecture를 sample하고 각 position별 marginal 분포를 확인. REINFORCE가 작동했다면 각 position의 분포가 ideal token에서 peak를 보여야 함.

In [ ]:
def diagnose_controller(controller: Controller, num_samples: int = 1000) -> np.ndarray:
    """Compute empirical marginal token distribution per position.

    Args:
        controller: A trained Controller instance.
        num_samples: Number of architectures to draw.

    Returns:
        Array of shape [T, vocab_size] with empirical token frequencies.
    """
    counts = np.zeros((T, VOCAB_SIZE), dtype=np.float32)
    with torch.no_grad():
        for _ in range(num_samples):
            tokens, _ = controller.sample()
            for t, tok in enumerate(tokens):
                counts[t, tok] += 1
    return counts / num_samples


# Re-train one final controller for diagnosis (deterministic seed).
torch.manual_seed(0)
diag_controller = Controller()
diag_optimizer = torch.optim.Adam(diag_controller.parameters(), lr=5e-3)
diag_baseline = 0.0
for _ in range(400):
    log_probs_per_arch = []
    rewards = []
    for _ in range(4):
        tokens, log_probs = diag_controller.sample()
        r = synthetic_reward(tokens)
        rewards.append(r)
        log_probs_per_arch.append(log_probs.sum())
    rewards_tensor = torch.tensor(rewards, dtype=torch.float32)
    advantages = rewards_tensor - diag_baseline
    diag_baseline = 0.95 * diag_baseline + 0.05 * float(rewards_tensor.mean().item())
    loss = -(torch.stack(log_probs_per_arch) * advantages).mean()
    diag_optimizer.zero_grad()
    loss.backward()
    diag_optimizer.step()

marginals = diagnose_controller(diag_controller, num_samples=1000)
ideal_full = []
for layer in range(NUM_LAYERS):
    ideal_full.append(IDEAL_SIZES_IDX[layer])
    ideal_full.append(IDEAL_COUNTS_IDX[layer])

fig, ax = plt.subplots(figsize=(11, 5))
im = ax.imshow(marginals, aspect='auto', cmap='viridis', vmin=0, vmax=1)
ax.set_xticks(range(VOCAB_SIZE))
ax.set_xticklabels([f'idx {i}' for i in range(VOCAB_SIZE)])
ax.set_yticks(range(T))
ax.set_yticklabels([f't={t} (ideal idx={ideal_full[t]})' for t in range(T)])
ax.set_xlabel('Token index')
ax.set_title('Controller marginal probabilities per position / 위치별 컨트롤러 주변 확률')
for t in range(T):
    ax.scatter(ideal_full[t], t, marker='*', s=180, c='red', edgecolor='white', linewidths=1, label='ideal' if t == 0 else None)
ax.legend(loc='upper right')
plt.colorbar(im, ax=ax, label='Empirical probability')
plt.tight_layout()
plt.show()

print('Per-position empirical probabilities:')
for t in range(T):
    bar = ' '.join(f'{p:.2f}' for p in marginals[t])
    print(f'  t={t}  ideal_idx={ideal_full[t]}  marginals=[{bar}]')

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Controller / 컨트롤러 | 2-layer LSTM (35 hidden) sampling 8+ tokens per arch | Transformer-based controller in NAS-Bench, AlphaCode-style structure prediction |
| Search space / 탐색 공간 | Macro CNN tokens or base-8 cell tree (~$6\times10^{16}$) | Cell-based DAG (NASNet/DARTS); hierarchical (MNASNet) |
| Reward / 보상 | Validation accuracy cubed (CIFAR), $80/\text{pp}^2$ (PTB) | Multi-objective: accuracy + latency (MNASNet) + energy (FBNet) |
| RL algorithm / RL 알고리즘 | REINFORCE + EMA baseline | PPO (NASNet variants), differentiable softmax (DARTS) |
| Compute / 연산 | 800 GPUs × 28 days ≈ 22,400 GPU-days | ENAS 0.45 GPU-day; DARTS 1 GPU-day; weight-sharing supernets |
| Training each child / 자식 학습 | Full 50 epochs | One-shot via shared weights or proxy tasks |
| Evaluation / 평가 | Single-task held-out val | NAS-Bench tabular benchmarks (101, 201, 301) |
| Headline result / 대표 결과 | CIFAR-10 3.65%, PTB pp 62.4 | EfficientNet-B7 84.3% top-1 ImageNet (AutoML lineage) |

**Key takeaways from this notebook / 노트북에서의 핵심 학습:**
1. The full NAS pipeline reduces to four ingredients: a search space, an auto-regressive controller, a (non-differentiable) reward, and a REINFORCE update.
2. With even a tiny 4-layer search space and a synthetic reward, REINFORCE provably moves the controller toward the optimum while random search is flat — exactly matching the paper's Figure 6 message.
3. The marginal-probability heatmap shows the controller concentrates probability mass on the ideal token at each position — direct visual evidence the policy gradient is doing its job.
4. NAS의 핵심 4요소(탐색 공간, auto-regressive controller, 미분 불가 보상, REINFORCE 업데이트)를 작은 환경에서도 동일하게 재현 가능하다.